# Gold - Modelos analiticos para dashboards de Power BI

Este notebook consume el modelo estrella Gold existente y genera tres tablas de resultados:

- `fact_pronostico_demanda.parquet`: evaluacion temporal de viajes reales frente a predichos.
- `fact_segmentacion_zonas.parquet`: segmentos de zonas de recojo mediante K-Means.
- `fact_clasificacion_demanda.parquet`: clasificacion de demanda baja, media o alta.

Los datos se filtran antes de entrenar. Solo se usan los meses entre 2023 y 2025, por lo que las fechas anomalas anteriores a 2023 y posteriores a 2025 no ingresan a ningun modelo.

## 1. Entorno y rutas

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os
import re
import subprocess
import sys
import urllib.request


def find_project_root(start=None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        fact_path = candidate / 'data' / 'gold' / 'modelo_estrella' / 'hechos' / 'fact_viajes_agregados.parquet'
        time_path = candidate / 'data' / 'gold' / 'modelo_estrella' / 'dimensiones' / 'dim_tiempo.parquet'
        if fact_path.exists() and time_path.exists():
            return candidate
    raise FileNotFoundError(
        'No se encontro la raiz con fact_viajes_agregados.parquet y dim_tiempo.parquet. '
        'Ejecuta primero modelo_estrella_parquet_powerbi.ipynb.'
    )


PROJECT_ROOT = find_project_root()
GOLD_ROOT = PROJECT_ROOT / 'data' / 'gold' / 'modelo_estrella'
DIMENSIONS_DIR = GOLD_ROOT / 'dimensiones'
FACTS_DIR = GOLD_ROOT / 'hechos'
SOURCE_FACT_PATH = FACTS_DIR / 'fact_viajes_agregados.parquet'
TIME_DIM_PATH = DIMENSIONS_DIR / 'dim_tiempo.parquet'
FACT_FORECAST_PATH = FACTS_DIR / 'fact_pronostico_demanda.parquet'
FACT_SEGMENTATION_PATH = FACTS_DIR / 'fact_segmentacion_zonas.parquet'
FACT_CLASSIFICATION_PATH = FACTS_DIR / 'fact_clasificacion_demanda.parquet'
FACTS_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_HADOOP_HOME = Path('C:/hadoop')
LOCAL_HADOOP_HOME = SYSTEM_HADOOP_HOME if SYSTEM_HADOOP_HOME.exists() else (PROJECT_ROOT / '.hadoop').resolve()
LOCAL_HADOOP_BIN = LOCAL_HADOOP_HOME / 'bin'
LOCAL_HADOOP_BIN.mkdir(parents=True, exist_ok=True)
WINUTILS_BASE_URL = 'https://raw.githubusercontent.com/steveloughran/winutils/master/hadoop-3.0.0/bin/'
for hadoop_file in ['winutils.exe', 'hadoop.dll']:
    target = LOCAL_HADOOP_BIN / hadoop_file
    if not target.exists():
        print(f'Descargando {hadoop_file} para PySpark en Windows...')
        urllib.request.urlretrieve(WINUTILS_BASE_URL + hadoop_file, target)

os.environ['HADOOP_HOME'] = LOCAL_HADOOP_HOME.as_posix()
os.environ['hadoop.home.dir'] = LOCAL_HADOOP_HOME.as_posix()
os.environ['PATH'] = str(LOCAL_HADOOP_BIN) + os.pathsep + os.environ.get('PATH', '')


def java_major_version():
    try:
        result = subprocess.run(['java', '-version'], capture_output=True, text=True, check=False)
        match = re.search(r'version "(\d+)', result.stderr or result.stdout)
        return int(match.group(1)) if match else None
    except Exception:
        return None


def find_jdk17_home():
    for root in [Path('C:/Program Files/Eclipse Adoptium'), Path('C:/Program Files/Java')]:
        if root.exists():
            for candidate in root.glob('**/jdk-17*'):
                if (candidate / 'bin' / 'java.exe').exists():
                    return candidate
    return None


if java_major_version() not in (None, 17):
    jdk17_home = find_jdk17_home()
    if jdk17_home is None:
        raise RuntimeError('Instala JDK 17 para ejecutar PySpark correctamente.')
    os.environ['JAVA_HOME'] = str(jdk17_home)
    os.environ['PATH'] = str(jdk17_home / 'bin') + os.pathsep + os.environ.get('PATH', '')

print('Proyecto:', PROJECT_ROOT)
print('Hechos de entrada:', SOURCE_FACT_PATH)
print('Salidas de modelos:', FACTS_DIR)

Proyecto: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos
Hechos de entrada: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos\fact_viajes_agregados.parquet
Salidas de modelos: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos


In [2]:
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import ClusteringEvaluator, MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import RandomForestRegressor
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--driver-memory 4g '
    f'--conf spark.driver.extraJavaOptions=-Dhadoop.home.dir={LOCAL_HADOOP_HOME.as_posix()} '
    f'--conf spark.executor.extraJavaOptions=-Dhadoop.home.dir={LOCAL_HADOOP_HOME.as_posix()} '
    'pyspark-shell'
)

spark = (
    SparkSession.builder
    .master('local[2]')
    .appName('Gold_Modelos_Dashboards_TLC')
    .config('spark.ui.enabled', 'false')
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .config('spark.driver.maxResultSize', '512m')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '64')
    .config('spark.sql.parquet.columnarReaderBatchSize', '1024')
    .config('spark.driver.extraJavaOptions', f'-Dhadoop.home.dir={LOCAL_HADOOP_HOME.as_posix()}')
    .config('spark.executor.extraJavaOptions', f'-Dhadoop.home.dir={LOCAL_HADOOP_HOME.as_posix()}')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark master:', spark.sparkContext.master)

Spark master: local[2]


## 2. Parametros y filtro de calidad temporal

`MIN_YEAR` y `MAX_YEAR` delimitan los datos validos del proyecto. El filtro se aplica mediante `dim_tiempo` antes de crear las variables de los modelos.

In [3]:
MIN_YEAR = 2023
MAX_YEAR = 2025
TEST_YEAR = 2025
K_CLUSTERS = 4
RANDOM_SEED = 42
FORECAST_TREES = 80
CLASSIFICATION_TREES = 80
MODEL_RUN_UTC = datetime.now(timezone.utc).replace(tzinfo=None)

if not (MIN_YEAR < TEST_YEAR <= MAX_YEAR):
    raise ValueError('TEST_YEAR debe estar dentro del rango y dejar al menos un anio para entrenamiento.')

print(f'Rango valido: {MIN_YEAR}-{MAX_YEAR}')
print(f'Entrenamiento: {MIN_YEAR}-{TEST_YEAR - 1}')
print(f'Prueba y resultados para dashboards: {TEST_YEAR}-{MAX_YEAR}')

Rango valido: 2023-2025
Entrenamiento: 2023-2024
Prueba y resultados para dashboards: 2025-2025


In [4]:
dim_tiempo_completa = spark.read.parquet(str(TIME_DIM_PATH))
periodos_excluidos = (
    dim_tiempo_completa
    .filter((F.col('anio') < MIN_YEAR) | (F.col('anio') > MAX_YEAR))
    .select('time_id', 'anio', 'mes')
    .orderBy('time_id')
)
dim_tiempo_valida = (
    dim_tiempo_completa
    .filter(F.col('anio').between(MIN_YEAR, MAX_YEAR))
    .select('time_id', 'fecha_inicio_mes', 'anio', 'mes', 'trimestre', 'orden_mes')
    .dropDuplicates(['time_id'])
)

print('Periodos excluidos por calidad temporal:')
periodos_excluidos.show(100, truncate=False)

fact_original = spark.read.parquet(str(SOURCE_FACT_PATH))
fact_modelos = (
    fact_original
    .join(F.broadcast(dim_tiempo_valida), 'time_id', 'inner')
    .persist(StorageLevel.DISK_ONLY)
)

filas_validas = fact_modelos.count()
anios_usados = [row['anio'] for row in fact_modelos.select('anio').distinct().orderBy('anio').collect()]
if not filas_validas:
    raise RuntimeError('No hay filas dentro del rango temporal configurado.')
if any(year < MIN_YEAR or year > MAX_YEAR for year in anios_usados):
    raise ValueError(f'El filtro temporal fallo. Anios observados: {anios_usados}')

print(f'Filas agregadas validas para modelado: {filas_validas:,}')
print('Anios usados por los modelos:', anios_usados)

Periodos excluidos por calidad temporal:
+-------+----+---+
|time_id|anio|mes|
+-------+----+---+
|2001-01|2001|1  |
|2002-12|2002|12 |
|2003-01|2003|1  |
|2007-12|2007|12 |
|2008-12|2008|12 |
|2009-01|2009|1  |
|2014-11|2014|11 |
|2022-10|2022|10 |
|2022-12|2022|12 |
|2026-01|2026|1  |
|2026-06|2026|6  |
+-------+----+---+

Filas agregadas validas para modelado: 13,699,385
Anios usados por los modelos: [2023, 2024, 2025]


## 3. Modelo de serie de tiempo

Se construye una serie mensual por tipo de taxi y zona de recojo. Un `RandomForestRegressor` usa tendencia, estacionalidad y rezagos de 1, 2 y 3 meses. Se entrena con 2023-2024 y se evalua sobre 2025.

In [5]:
serie_mensual = (
    fact_modelos
    .groupBy('time_id', 'fecha_inicio_mes', 'anio', 'mes', 'trimestre', 'orden_mes', 'tipo_dataset', 'pickup_location_id')
    .agg(F.sum('cantidad_viajes').cast('double').alias('viajes_reales'))
)

ventana_serie = Window.partitionBy('tipo_dataset', 'pickup_location_id').orderBy('orden_mes')
ventana_media = ventana_serie.rowsBetween(-3, -1)
serie_features = (
    serie_mensual
    .withColumn('viajes_lag_1', F.lag('viajes_reales', 1).over(ventana_serie))
    .withColumn('viajes_lag_2', F.lag('viajes_reales', 2).over(ventana_serie))
    .withColumn('viajes_lag_3', F.lag('viajes_reales', 3).over(ventana_serie))
    .withColumn('media_movil_3', F.avg('viajes_reales').over(ventana_media))
    .filter(F.col('viajes_lag_3').isNotNull())
)

train_serie = serie_features.filter(F.col('anio') < TEST_YEAR)
test_serie = serie_features.filter(F.col('anio') >= TEST_YEAR)
if train_serie.limit(1).count() == 0 or test_serie.limit(1).count() == 0:
    raise RuntimeError('La serie temporal necesita filas de entrenamiento y prueba.')

tipo_indexer_reg = StringIndexer(inputCol='tipo_dataset', outputCol='tipo_dataset_idx', handleInvalid='keep')
tipo_encoder_reg = OneHotEncoder(inputCols=['tipo_dataset_idx'], outputCols=['tipo_dataset_vec'], handleInvalid='keep')
assembler_reg = VectorAssembler(
    inputCols=[
        'orden_mes', 'mes', 'trimestre', 'pickup_location_id',
        'viajes_lag_1', 'viajes_lag_2', 'viajes_lag_3', 'media_movil_3', 'tipo_dataset_vec',
    ],
    outputCol='features',
    handleInvalid='skip',
)
regresor = RandomForestRegressor(
    featuresCol='features', labelCol='viajes_reales', predictionCol='viajes_predichos',
    numTrees=FORECAST_TREES, maxDepth=10, subsamplingRate=0.8, seed=RANDOM_SEED,
)
pipeline_regresion = Pipeline(stages=[tipo_indexer_reg, tipo_encoder_reg, assembler_reg, regresor])
modelo_regresion = pipeline_regresion.fit(train_serie)
predicciones_serie = modelo_regresion.transform(test_serie)

rmse = RegressionEvaluator(labelCol='viajes_reales', predictionCol='viajes_predichos', metricName='rmse').evaluate(predicciones_serie)
mae = RegressionEvaluator(labelCol='viajes_reales', predictionCol='viajes_predichos', metricName='mae').evaluate(predicciones_serie)
r2 = RegressionEvaluator(labelCol='viajes_reales', predictionCol='viajes_predichos', metricName='r2').evaluate(predicciones_serie)

fact_pronostico_demanda = (
    predicciones_serie
    .withColumn('viajes_predichos', F.greatest(F.col('viajes_predichos'), F.lit(0.0)))
    .withColumn('error_absoluto', F.abs(F.col('viajes_reales') - F.col('viajes_predichos')))
    .withColumn(
        'error_porcentual',
        F.when(F.col('viajes_reales') > 0, F.col('error_absoluto') / F.col('viajes_reales') * 100.0),
    )
    .withColumn('modelo', F.lit('RandomForestRegressor con rezagos'))
    .withColumn('rmse_modelo', F.lit(float(rmse)))
    .withColumn('mae_modelo', F.lit(float(mae)))
    .withColumn('r2_modelo', F.lit(float(r2)))
    .withColumn('fecha_generacion_utc', F.lit(MODEL_RUN_UTC).cast('timestamp'))
    .select(
        'time_id', 'tipo_dataset', 'pickup_location_id', 'viajes_reales', 'viajes_predichos',
        'error_absoluto', 'error_porcentual', 'modelo', 'rmse_modelo', 'mae_modelo',
        'r2_modelo', 'fecha_generacion_utc',
    )
)
fact_pronostico_demanda.coalesce(1).write.mode('overwrite').parquet(str(FACT_FORECAST_PATH))
print(f'Pronostico guardado: {FACT_FORECAST_PATH}')
print(f'RMSE={rmse:,.2f} | MAE={mae:,.2f} | R2={r2:.4f}')
fact_pronostico_demanda.show(10, truncate=False)

Pronostico guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos\fact_pronostico_demanda.parquet
RMSE=28,278.85 | MAE=2,229.03 | R2=0.9241
+-------+------------+------------------+-------------+------------------+-----------------+------------------+---------------------------------+------------------+------------------+------------------+--------------------------+
|time_id|tipo_dataset|pickup_location_id|viajes_reales|viajes_predichos  |error_absoluto   |error_porcentual  |modelo                           |rmse_modelo       |mae_modelo        |r2_modelo         |fecha_generacion_utc      |
+-------+------------+------------------+-------------+------------------+-----------------+------------------+---------------------------------+------------------+------------------+------------------+--------------------------+
|2025-01|fhv         |-1                |1858437.0    |1445087

## 4. Modelo de segmentacion

K-Means agrupa las zonas de recojo por volumen, ingresos, distancia, duracion y ticket promedio. Cada fila representa el perfil de una zona para un tipo de taxi.

In [6]:
perfil_zonas = (
    fact_modelos
    .groupBy('tipo_dataset', 'pickup_location_id')
    .agg(
        F.sum('cantidad_viajes').cast('double').alias('cantidad_viajes'),
        F.sum('total_monto_total').cast('double').alias('ingreso_total'),
        F.sum('total_distancia_millas').cast('double').alias('distancia_total'),
        F.sum('total_duracion_minutos').cast('double').alias('duracion_total'),
        F.countDistinct('time_id').alias('meses_con_datos'),
    )
    .filter(F.col('cantidad_viajes') > 0)
    .withColumn('viajes_por_mes', F.col('cantidad_viajes') / F.col('meses_con_datos'))
    .withColumn('ticket_promedio', F.col('ingreso_total') / F.col('cantidad_viajes'))
    .withColumn('distancia_promedio', F.col('distancia_total') / F.col('cantidad_viajes'))
    .withColumn('duracion_promedio', F.col('duracion_total') / F.col('cantidad_viajes'))
    .fillna(0.0, subset=['ingreso_total', 'viajes_por_mes', 'ticket_promedio', 'distancia_promedio', 'duracion_promedio'])
    .withColumn('log_cantidad_viajes', F.log1p('cantidad_viajes'))
    .withColumn('log_ingreso_total', F.log1p(F.greatest(F.col('ingreso_total'), F.lit(0.0))))
)
if perfil_zonas.count() < K_CLUSTERS:
    raise RuntimeError(f'K-Means necesita al menos {K_CLUSTERS} perfiles de zona.')

assembler_kmeans = VectorAssembler(
    inputCols=[
        'log_cantidad_viajes', 'log_ingreso_total', 'viajes_por_mes',
        'ticket_promedio', 'distancia_promedio', 'duracion_promedio',
    ],
    outputCol='features_sin_escalar',
    handleInvalid='skip',
)
scaler_kmeans = StandardScaler(
    inputCol='features_sin_escalar', outputCol='features_escaladas', withStd=True, withMean=True
)
kmeans = KMeans(
    k=K_CLUSTERS, seed=RANDOM_SEED, featuresCol='features_escaladas', predictionCol='cluster_id'
)
pipeline_kmeans = Pipeline(stages=[assembler_kmeans, scaler_kmeans, kmeans])
modelo_kmeans = pipeline_kmeans.fit(perfil_zonas)
segmentos = modelo_kmeans.transform(perfil_zonas)
silhouette = ClusteringEvaluator(
    featuresCol='features_escaladas', predictionCol='cluster_id', metricName='silhouette'
).evaluate(segmentos)

fact_segmentacion_zonas = (
    segmentos
    .withColumn('cluster_id', F.col('cluster_id').cast('int'))
    .withColumn('segmento', F.concat(F.lit('Segmento '), F.col('cluster_id').cast('string')))
    .withColumn('modelo', F.lit(f'KMeans k={K_CLUSTERS}'))
    .withColumn('silhouette_modelo', F.lit(float(silhouette)))
    .withColumn('fecha_generacion_utc', F.lit(MODEL_RUN_UTC).cast('timestamp'))
    .select(
        'tipo_dataset', 'pickup_location_id', 'cluster_id', 'segmento', 'cantidad_viajes',
        'ingreso_total', 'viajes_por_mes', 'ticket_promedio', 'distancia_promedio',
        'duracion_promedio', 'meses_con_datos', 'modelo', 'silhouette_modelo', 'fecha_generacion_utc',
    )
)
fact_segmentacion_zonas.coalesce(1).write.mode('overwrite').parquet(str(FACT_SEGMENTATION_PATH))
print(f'Segmentacion guardada: {FACT_SEGMENTATION_PATH}')
print(f'Silhouette={silhouette:.4f}')
fact_segmentacion_zonas.orderBy('cluster_id', F.desc('cantidad_viajes')).show(20, truncate=False)

Segmentacion guardada: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos\fact_segmentacion_zonas.parquet
Silhouette=0.4827
+------------+------------------+----------+----------+---------------+------------------+------------------+------------------+------------------+------------------+---------------+----------+------------------+--------------------------+
|tipo_dataset|pickup_location_id|cluster_id|segmento  |cantidad_viajes|ingreso_total     |viajes_por_mes    |ticket_promedio   |distancia_promedio|duracion_promedio |meses_con_datos|modelo    |silhouette_modelo |fecha_generacion_utc      |
+------------+------------------+----------+----------+---------------+------------------+------------------+------------------+------------------+------------------+---------------+----------+------------------+--------------------------+
|green       |193               |0         |Segmento 0

## 5. Modelo de clasificacion

La demanda por mes, hora, tipo de taxi y zona se clasifica como baja, media o alta. Los umbrales se calculan solamente con el periodo de entrenamiento para evitar fuga de informacion desde 2025.

In [7]:
demanda_contexto = (
    fact_modelos
    .groupBy('time_id', 'anio', 'mes', 'trimestre', 'hora_id', 'tipo_dataset', 'pickup_location_id')
    .agg(F.sum('cantidad_viajes').cast('double').alias('cantidad_viajes'))
    .filter(F.col('hora_id').between(0, 23) & (F.col('pickup_location_id') >= 0))
)
demanda_entrenamiento_base = demanda_contexto.filter(F.col('anio') < TEST_YEAR)
quantiles = demanda_entrenamiento_base.approxQuantile('cantidad_viajes', [0.33, 0.66], 0.01)
if len(quantiles) != 2:
    raise RuntimeError('No se pudieron calcular los umbrales de clasificacion.')
umbral_bajo, umbral_alto = [float(value) for value in quantiles]
if umbral_alto <= umbral_bajo:
    umbral_alto = umbral_bajo + 1.0

demanda_clasificada = (
    demanda_contexto
    .withColumn(
        'clase_demanda_id',
        F.when(F.col('cantidad_viajes') <= umbral_bajo, F.lit(0.0))
        .when(F.col('cantidad_viajes') <= umbral_alto, F.lit(1.0))
        .otherwise(F.lit(2.0)),
    )
)
train_clasificacion = demanda_clasificada.filter(F.col('anio') < TEST_YEAR)
test_clasificacion = demanda_clasificada.filter(F.col('anio') >= TEST_YEAR)
if train_clasificacion.limit(1).count() == 0 or test_clasificacion.limit(1).count() == 0:
    raise RuntimeError('La clasificacion necesita filas de entrenamiento y prueba.')

tipo_indexer_clf = StringIndexer(inputCol='tipo_dataset', outputCol='tipo_dataset_idx', handleInvalid='keep')
tipo_encoder_clf = OneHotEncoder(inputCols=['tipo_dataset_idx'], outputCols=['tipo_dataset_vec'], handleInvalid='keep')
assembler_clf = VectorAssembler(
    inputCols=['mes', 'trimestre', 'hora_id', 'pickup_location_id', 'tipo_dataset_vec'],
    outputCol='features',
    handleInvalid='skip',
)
clasificador = RandomForestClassifier(
    featuresCol='features', labelCol='clase_demanda_id', predictionCol='clase_predicha_id',
    probabilityCol='probability', numTrees=CLASSIFICATION_TREES, maxDepth=12,
    subsamplingRate=0.8, seed=RANDOM_SEED,
)
pipeline_clasificacion = Pipeline(stages=[tipo_indexer_clf, tipo_encoder_clf, assembler_clf, clasificador])
modelo_clasificacion = pipeline_clasificacion.fit(train_clasificacion)
predicciones_clasificacion = modelo_clasificacion.transform(test_clasificacion)
accuracy = MulticlassClassificationEvaluator(
    labelCol='clase_demanda_id', predictionCol='clase_predicha_id', metricName='accuracy'
).evaluate(predicciones_clasificacion)
f1 = MulticlassClassificationEvaluator(
    labelCol='clase_demanda_id', predictionCol='clase_predicha_id', metricName='f1'
).evaluate(predicciones_clasificacion)

def demand_label(column_name):
    value = F.col(column_name)
    return F.when(value == 0, F.lit('Baja')).when(value == 1, F.lit('Media')).otherwise(F.lit('Alta'))


fact_clasificacion_demanda = (
    predicciones_clasificacion
    .withColumn('clase_demanda_id', F.col('clase_demanda_id').cast('int'))
    .withColumn('clase_predicha_id', F.col('clase_predicha_id').cast('int'))
    .withColumn('demanda_real', demand_label('clase_demanda_id'))
    .withColumn('demanda_predicha', demand_label('clase_predicha_id'))
    .withColumn('probabilidad_predicha', F.array_max(vector_to_array('probability')))
    .withColumn('prediccion_correcta', F.col('clase_demanda_id') == F.col('clase_predicha_id'))
    .withColumn('modelo', F.lit('RandomForestClassifier'))
    .withColumn('accuracy_modelo', F.lit(float(accuracy)))
    .withColumn('f1_modelo', F.lit(float(f1)))
    .withColumn('umbral_bajo', F.lit(umbral_bajo))
    .withColumn('umbral_alto', F.lit(umbral_alto))
    .withColumn('fecha_generacion_utc', F.lit(MODEL_RUN_UTC).cast('timestamp'))
    .select(
        'time_id', 'hora_id', 'tipo_dataset', 'pickup_location_id', 'cantidad_viajes',
        'clase_demanda_id', 'demanda_real', 'clase_predicha_id', 'demanda_predicha',
        'probabilidad_predicha', 'prediccion_correcta', 'modelo', 'accuracy_modelo',
        'f1_modelo', 'umbral_bajo', 'umbral_alto', 'fecha_generacion_utc',
    )
)
fact_clasificacion_demanda.coalesce(1).write.mode('overwrite').parquet(str(FACT_CLASSIFICATION_PATH))
print(f'Clasificacion guardada: {FACT_CLASSIFICATION_PATH}')
print(f'Umbrales: baja <= {umbral_bajo:,.0f}; media <= {umbral_alto:,.0f}; alta > {umbral_alto:,.0f}')
print(f'Accuracy={accuracy:.4f} | F1={f1:.4f}')
fact_clasificacion_demanda.show(10, truncate=False)

Clasificacion guardada: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos\fact_clasificacion_demanda.parquet
Umbrales: baja <= 3; media <= 24; alta > 24
Accuracy=0.5225 | F1=0.5237
+-------+-------+------------+------------------+---------------+----------------+------------+-----------------+----------------+---------------------+-------------------+----------------------+------------------+------------------+-----------+-----------+--------------------------+
|time_id|hora_id|tipo_dataset|pickup_location_id|cantidad_viajes|clase_demanda_id|demanda_real|clase_predicha_id|demanda_predicha|probabilidad_predicha|prediccion_correcta|modelo                |accuracy_modelo   |f1_modelo         |umbral_bajo|umbral_alto|fecha_generacion_utc      |
+-------+-------+------------+------------------+---------------+----------------+------------+-----------------+----------------+----------------

## 6. Validacion de salidas

In [8]:
salidas = {
    'fact_pronostico_demanda': FACT_FORECAST_PATH,
    'fact_segmentacion_zonas': FACT_SEGMENTATION_PATH,
    'fact_clasificacion_demanda': FACT_CLASSIFICATION_PATH,
}
for nombre, ruta in salidas.items():
    resultado = spark.read.parquet(str(ruta))
    filas = resultado.count()
    if filas == 0:
        raise ValueError(f'{nombre} se genero sin filas.')
    print(f'{nombre}: {filas:,} filas -> {ruta}')

anios_resultados = (
    spark.read.parquet(str(FACT_FORECAST_PATH))
    .join(dim_tiempo_completa.select('time_id', 'anio'), 'time_id', 'left')
    .select('anio').distinct().orderBy('anio').collect()
)
if any(row['anio'] is None or row['anio'] < MIN_YEAR or row['anio'] > MAX_YEAR for row in anios_resultados):
    raise ValueError(f'El pronostico contiene periodos fuera del rango valido: {anios_resultados}')

fact_modelos.unpersist()
print('Modelos Gold publicados correctamente.')

fact_pronostico_demanda: 4,984 filas -> C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos\fact_pronostico_demanda.parquet
fact_segmentacion_zonas: 515 filas -> C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos\fact_segmentacion_zonas.parquet
fact_clasificacion_demanda: 78,637 filas -> C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\hechos\fact_clasificacion_demanda.parquet
Modelos Gold publicados correctamente.


## 7. Relaciones en Power BI

Carga las tres carpetas Parquet con **Obtener datos > Carpeta > Combinar y transformar**. Conserva las dimensiones actuales y agrega estas relaciones 1:*:

| Tabla de resultados | Relaciones |
|---|---|
| `fact_pronostico_demanda` | `dim_tiempo[time_id]`, `dim_tipo_taxi[tipo_dataset]`, `dim_zona_pickup[pickup_location_id]` |
| `fact_segmentacion_zonas` | `dim_tipo_taxi[tipo_dataset]`, `dim_zona_pickup[pickup_location_id]` |
| `fact_clasificacion_demanda` | `dim_tiempo[time_id]`, `dim_hora[hora_id]`, `dim_tipo_taxi[tipo_dataset]`, `dim_zona_pickup[pickup_location_id]` |

No relaciones las tablas de hechos entre si. Todas deben compartir dimensiones. Elimina `Source.Name` en Power Query y comprueba que las claves tengan el mismo tipo de datos en ambos lados de cada relacion.

## 8. Orden de actualizacion

1. Ejecutar `modelo_estrella_parquet_powerbi.ipynb` para actualizar el modelo estrella base.
2. Ejecutar este notebook para reentrenar los modelos y reemplazar sus tres tablas de resultados.
3. Actualizar el modelo semantico de Power BI manualmente o mediante el Gateway.

Estas tablas se reconstruyen completas porque un modelo nuevo puede cambiar la prediccion o el segmento de registros anteriores.